# 3.14 - Debogage d un entrainement qui tourne mais n apprend pas
**Navigation** : [<< 3.13b - Deux vraies cartes : DDP et FSDP sous nccl](3.13b-Deux-Cartes-Reelles-DDP-FSDP-nccl.ipynb) · [Feuille de route de la serie](README.md)

Carnet pedagogique du parcours 03-DeepLearning. Il vise a **diagnostiquer** un modele qui boucle sans faire baisser la perte : loss constante, loss qui oscille, loss qui explose, loss qui descend mais plafonne.

**Cinq controles** systematiques dans l ordre :
1. **Data pipeline** - X.shape, range, normalisation, NaN/Inf dans y.
2. **Loss correcte** - perte adaptee a la tache (regression / binaire / multi-classes).
3. **Gradient** - norme non-NaN, valeurs raisonnables (entre 0 et quelques dizaines).
4. **Learning rate** - ni trop petit (loss stagne), ni trop grand (loss oscille ou explose).
5. **Initialisation** - poids pas tous a zero, variance raisonnable.

Ce carnet utilise **`SGDClassifier`** de scikit-learn (CPU strict) pour montrer ces patterns sur des donnees jouet (classification binaire). La perte est mesuree explicitement par `log_loss(y, clf.predict_proba(X))` apres chaque passe -- `SGDClassifier` n expose pas de `loss_curve_`.

In [1]:
# %pip install -q scikit-learn numpy

import numpy as np
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import log_loss

rng = np.random.default_rng(seed=42)
n = 400
X = rng.normal(size=(n, 2))
y = (X[:, 0] + X[:, 1] > 0).astype(int)
print(f"X.shape={X.shape}, y.shape={y.shape}, classes={np.unique(y).tolist()}")
print(f"X min/max: {X.min():.2f} / {X.max():.2f}")
print(f"NaN dans X: {np.isnan(X).sum()}, NaN dans y: {np.isnan(y).sum()}")

X.shape=(400, 2), y.shape=(400,), classes=[0, 1]
X min/max: -2.96 / 2.91
NaN dans X: 0, NaN dans y: 0


## 1. Quand la perte stagne - lr trop petit

**Symptome** - la perte ne bouge pas d une epoch a l autre (ou descend si lentement que le budget d epochs est insuffisant).

**Cause typique** - `learning_rate` initial trop petit (`eta0=1e-6`) ou `learning_rate="optimal"` avec une regularisation trop forte.

In [2]:
# Pattern 1 - loss stagne avec eta0 = 1e-6
clf = SGDClassifier(
    loss="log_loss",
    learning_rate="constant",
    eta0=1e-6,
    max_iter=50,
    tol=None,
    random_state=42,
)
losses_stagne = []
for _ in range(50):
    clf.partial_fit(X, y, classes=[0, 1])
    losses_stagne.append(log_loss(y, clf.predict_proba(X)))

print(f"loss initiale : {losses_stagne[0]:.6f}")
print(f"loss finale   : {losses_stagne[-1]:.6f}")
print(f"evolution     : {losses_stagne[0] - losses_stagne[-1]:.2e}")

sample = [losses_stagne[i] for i in (0, 10, 20, 30, 40, 49)]
for i, e in zip([0, 10, 20, 30, 40, 49], sample):
    print(f"  epoch {i:2d} - loss = {e:.6f}")

loss initiale : 0.693075
loss finale   : 0.689577
evolution     : 3.50e-03
  epoch  0 - loss = 0.693075
  epoch 10 - loss = 0.692358
  epoch 20 - loss = 0.691643
  epoch 30 - loss = 0.690929
  epoch 40 - loss = 0.690217
  epoch 49 - loss = 0.689577


**Lecture du resultat.** La perte va de 0.6931 a 0.6896 en 50 passes - elle reste collee au **niveau du hasard** (ln 2 = 0.693, le pile-ou-face). Le modele n apprend **rien**.

**Diagnostic.** Premier reflexe - verifier le **learning rate**. Ici `eta0=1e-6` est 4 ordres de grandeur en dessous du lr de reference (0.01) - le modele ne peut pas bouger.

In [3]:
# Exercice 1 - reproduire le pattern loss stagne avec un modele lineaire simple.
#
# Objectif - trouver un learning_rate et un nombre d epochs tels que la perte
# reste quasi constante (< 1e-4 de variation) sur SGDClassifier.
#
# Indices -
# - learning_rate="constant", eta0 petit (10^-7, 10^-8)
# - max_iter=20-30
# - tol=None
# - affichez loss initiale, loss finale, et l evolution

pass  # Remplacez par votre code (regle C.1 - pas de raise, juste pass / print)

## 2. Quand la perte oscille ou explose - lr trop grand

**Symptome** - la perte saute d une passe a l autre sans se stabiliser (oscillation), ou depasse 1 : le modele devient confiant dans ses erreurs.

**Cause typique** - `learning_rate` initial trop grand (`eta0=10.0` et au-dela) : chaque pas de gradient franchit la vallee, les poids passent d un cote puis de l autre, et la perte ne s installe jamais.

In [4]:
# Pattern 2 - loss instable avec eta0 = 10.0 (oscille sans converger)
clf2 = SGDClassifier(
    loss="log_loss",
    learning_rate="constant",
    eta0=10.0,
    max_iter=10,
    tol=None,
    random_state=42,
)
losses_explose = []
for _ in range(10):
    try:
        clf2.partial_fit(X, y, classes=[0, 1])
        losses_explose.append(log_loss(y, clf2.predict_proba(X)))
    except Exception as e:
        losses_explose.append(float("nan"))

print(f"loss epoch 0 - {losses_explose[0]:.4f}")
print(f"loss epoch 5 - {losses_explose[5]:.4f}")
print(f"loss epoch 9 - {losses_explose[9]:.4f}")

import math
finite = [l for l in losses_explose if math.isfinite(l)]
print(f"epochs finis - {len(finite)} / {len(losses_explose)}")
if finite:
    print(f"loss max (finie) - {max(finite):.4e}")

loss epoch 0 - 0.6612


loss epoch 5 - 0.2422
loss epoch 9 - 0.5280
epochs finis - 10 / 10
loss max (finie) - 7.4906e-01


**Lecture du resultat.** La perte oscille - 0.66, puis 0.24, puis 0.53 - sans jamais se stabiliser : ce n est pas une convergence plus lente, c est une **trajectoire instable**. Avec un lr encore plus grand (`eta0=50`, voir le diagnostic ci-dessous), la perte depasse 1 : le modele se trompe **avec confiance**.

**Diagnostic.** Premier reflexe - reduire `eta0` d un facteur 10 a 1000. L alternative `penalty=elasticnet` ou `learning_rate="adaptive"` (sklearn) adoucit la trajectoire.

In [5]:
# Exercice 2 - reproduire le pattern loss instable et trouver le seuil d instabilite.
#
# Objectif - pour eta0 variant entre 0.1 et 50.0, observer a partir de quel eta0
# la perte devient instable sur SGDClassifier(log_loss, learning_rate="constant") :
# oscillations entre passes (saut > 0.1 entre deux epochs consecutives) ou
# perte finale > 1 (le modele se trompe avec confiance).
#
# Sortie attendue - un tableau textuel eta0 vers perte_finale.

pass  # Remplacez par votre code

## 3. Quand la perte descend bien - lr bien calibre

**Pattern de reference**. Avec `eta0=0.01`, la perte quitte le niveau du hasard des la premiere passe (0.69 -> 0.38), puis descend jusqu a environ 0.11 en 30 passes.

In [6]:
# Pattern 3 - loss descend bien
clf3 = SGDClassifier(
    loss="log_loss",
    learning_rate="constant",
    eta0=0.01,
    max_iter=30,
    tol=None,
    random_state=42,
)
losses_ok = []
for _ in range(30):
    clf3.partial_fit(X, y, classes=[0, 1])
    losses_ok.append(log_loss(y, clf3.predict_proba(X)))

print(f"loss initiale - {losses_ok[0]:.4f}")
print(f"loss finale   - {losses_ok[-1]:.4f}")
print(f"evolution     - {losses_ok[0] - losses_ok[-1]:+.4f}")

print()
print("Comparaison 3 patterns -")
print(f"  Pattern 1 (lr trop petit)   - loss finale = {losses_stagne[-1]:.4f}")
print(f"  Pattern 2 (lr trop grand)   - loss finale = {losses_explose[-1]:.4e}")
print(f"  Pattern 3 (lr bien calibre) - loss finale = {losses_ok[-1]:.4f}")

loss initiale - 0.3788
loss finale   - 0.1107
evolution     - +0.2681

Comparaison 3 patterns -
  Pattern 1 (lr trop petit)   - loss finale = 0.6896
  Pattern 2 (lr trop grand)   - loss finale = 5.2797e-01
  Pattern 3 (lr bien calibre) - loss finale = 0.1107


**Lecture du resultat.** Des la premiere passe la perte tombe de 0.69 (hasard) a 0.38, puis elle descend jusqu a 0.11 apres 30 passes - le modele apprend.

**Pourquoi `eta0=0.01` marche ?** L echelle du learning rate doit correspondre a l echelle des gradients (lui-meme dependant de l echelle des features). Avec des features en `~N(0, 1)`, eta0=0.01 est dans la bonne fenetre.

## 4. Diagnostic en 5 checks

Quand l entrainement ne baisse pas, executer ces 5 checks **dans l ordre** (du plus rapide au plus lent) -

| # | Question | Verification |
|---|---|---|
| 1 | Data pipeline | X.shape, range, normalisation, NaN/Inf dans y |
| 2 | Loss correcte | perte adaptee a la tache (pas de MSE sur classification multi-classes) |
| 3 | Gradient | norme non-NaN, valeurs raisonnables (entre 0 et quelques dizaines) |
| 4 | Learning rate | ni trop petit (stagne), ni trop grand (oscille ou explose) |
| 5 | Initialisation | poids pas tous a zero, variance raisonnable |

Mini-outil ci-dessous implemente les checks 1 et 4 (les 3 autres dependent du framework).

In [7]:
def diagnostic_data(X, y):
    """Check 1 - data pipeline."""
    msgs = []
    if np.isnan(X).any():
        msgs.append(f"KO - X contient {np.isnan(X).sum()} NaN")
    if np.isnan(y).any():
        msgs.append(f"KO - y contient {np.isnan(y).sum()} NaN")
    if np.isinf(X).any() or np.isinf(y).any():
        msgs.append("KO - X ou y a des Inf")
    if X.std() < 1e-6 or X.std() > 100:
        msgs.append(f"WARN - X.std()={X.std():.4f} (range attendu ~1)")
    if not msgs:
        msgs.append(f"OK - X.shape={X.shape}, X.std={X.std():.3f}, NaN=0")
    return msgs

def diagnostic_lr(eta0, n_iter):
    """Check 4 - learning rate sur SGDClassifier(log_loss)."""
    clf = SGDClassifier(loss="log_loss", learning_rate="constant", eta0=eta0,
                        max_iter=n_iter, tol=None, random_state=42)
    losses = []
    for _ in range(n_iter):
        clf.partial_fit(X, y, classes=[0, 1])
        losses.append(log_loss(y, clf.predict_proba(X)))
    fin = losses[-1]
    saut_max = max(abs(b - a) for a, b in zip(losses, losses[1:]))
    if fin > 1.0:
        return [f"KO - lr={eta0} -> explose (perte finale {fin:.2f} > 1 : le modele se trompe avec confiance)"]
    if saut_max > 0.1:
        return [f"KO - lr={eta0} -> oscille sans converger (saut max {saut_max:.2f} entre deux passes)"]
    if fin > 0.68:
        return [f"KO - lr={eta0} -> stagne (perte finale {fin:.4f}, au voisinage du hasard ln(2) = 0.693)"]
    return [f"OK - lr={eta0} -> converge (perte finale {fin:.4f})"]

print("=== Diagnostic data ===")
for msg in diagnostic_data(X, y):
    print(f"  {msg}")

print()
print("=== Diagnostic lr ===")
for eta in [1e-6, 0.01, 0.1, 1.0, 10.0, 50.0]:
    for msg in diagnostic_lr(eta, n_iter=30):
        print(f"  {msg}")

=== Diagnostic data ===
  OK - X.shape=(400, 2), X.std=0.984, NaN=0

=== Diagnostic lr ===


  KO - lr=1e-06 -> stagne (perte finale 0.6910, au voisinage du hasard ln(2) = 0.693)


  OK - lr=0.01 -> converge (perte finale 0.1107)
  OK - lr=0.1 -> converge (perte finale 0.0548)


  OK - lr=1.0 -> converge (perte finale 0.0357)
  KO - lr=10.0 -> oscille sans converger (saut max 0.54 entre deux passes)


  KO - lr=50.0 -> explose (perte finale 3.31 > 1 : le modele se trompe avec confiance)


**Lecture du resultat.**

- **Data** - X.std() proche de 1, pas de NaN/Inf -> pipeline OK.
- **lr** - `1e-6` trop petit (stagne au voisinage du hasard), `0.01` / `0.1` / `1.0` convergent, `10.0` trop grand (oscille sans converger), `50.0` trop grand (explose au-dela de 1).

**Conclusion.** Avec ces donnees jouet et `SGDClassifier(log_loss)`, la fenetre viable est environ `[0.001, 1]`. En dessous la perte stagne au niveau du hasard, au-dessus elle oscille puis explose.

In [8]:
# Exercice 3 - appliquer le diagnostic a un modele malade.
#
# Objectif - on vous donne un modele ci-dessous dont la perte stagne ou explose.
# A vous d utiliser diagnostic_data() et diagnostic_lr() pour identifier la cause.
#
# Sortie attendue - pour chaque lr teste, le verdict (OK / KO) et la cause probable.

# Votre code ici (regle C.1 - pas de raise)
pass

## 5. Conclusion

Quand un entrainement ne baisse pas, **les 5 checks sont un parcours**, pas une check-list - on avance dans l ordre, et chaque check KO doit vous orienter vers une hypothese precise.

Les patterns les plus frequents -

- **lr trop petit** - perte stagne au niveau du hasard. Solution - augmenter `eta0` d un facteur 10.
- **lr trop grand** - perte oscille puis explose au-dela de 1. Solution - reduire `eta0` ou ajouter du gradient clipping.
- **data pipeline** - NaN/Inf dans X ou y. Solution - verifier la source, `np.isnan(X).any()`.
- **loss incorrecte** - MSE sur classification. Solution - `log_loss` (binaire) ou  `categorical_crossentropy` (multi-classes).
- **mauvaise initialisation** - modele constant. Solution - Glorot/Xavier ou le default framework.

**Suite naturelle** - ce carnet ferme la sous-serie 03-DeepLearning. La [feuille de route de la serie](README.md) donne les parcours suivants ; `02-ML-Cours/2.13-Analyse-Erreurs.ipynb` est le pendant en ML classique.